# OCR demo — grant-administration documents

A walkthrough of the full extraction pipeline for university grant docs
(award notices, budgets, proposals) using a Qwen-VL vision-language model
deployed on RunAI.

The pipeline has three pieces:

1. **Render** each PDF page as an image (PyMuPDF, ~144 DPI).
2. **Extract** structured JSON from each image via a deployed vLLM endpoint
   running Qwen3-VL-32B. Pages are sent in chunks of up to ~20 with a 1-page
   sliding overlap so the model can reason about cross-page tables and
   narratives.
3. **Merge + synthesize** chunk outputs into one document-level JSON.
   Pass 2 adds a short doc-level summary and lints for issues that the
   per-chunk extraction can't see (continuation flags still set, possible
   duplicate stakeholders, etc.).

The shared pipeline lives in `ocr_app/scripts/` — this notebook drives it
interactively so you can watch each stage. The same pieces run unchanged
inside the production batch script (`ocr_app/scripts/batch_extract.py`).

> **Library / archival materials?** Use `library_extraction_pipeline_demo.ipynb` —
> same architecture, library-specific prompt and schema.


## 1. Connect to the deployed vLLM endpoint

The model runs as a separate RunAI workload exposing an OpenAI-compatible
HTTP endpoint. This notebook is just a CPU workspace that talks to it.


In [ ]:
import os
import httpx

VLLM_BASE_URL = os.environ.get(
    "VLLM_BASE_URL",
    "http://qwen3--vl--32b--instruct-awq.runai-shared-models.svc.cluster.local/v1",
)
# Per-call output budget. Sized to stay under the gateway's 10-min cap on
# dense 10–20-page chunks; lower if your endpoint has a tighter timeout.
VLLM_MAX_TOKENS = 90000

resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=10)
resp.raise_for_status()
MODEL_NAME = resp.json()["data"][0]["id"]
print(f"Endpoint: {VLLM_BASE_URL}")
print(f"Model:    {MODEL_NAME}")

# Smoke test — confirm the endpoint actually generates.
resp = httpx.post(
    f"{VLLM_BASE_URL}/chat/completions",
    json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "In a single sentence, share with me the entire life story of Winnie the Pooh."}],
        "max_tokens": 100,
        "temperature": 0.0,
    },
    timeout=60,
)
resp.raise_for_status()
print("Smoke test:", resp.json()["choices"][0]["message"]["content"].strip())


### Running this notebook from your laptop instead of a RunAI workspace?

The default URL above is the in-cluster service hostname — it only resolves
from inside the cluster. If you're on the campus VPN, point at the public
Knative hostname instead. Same OpenAI-compatible API, no API key required.

Here's a self-contained example using the `openai` Python client (run as-is
to confirm your VPN can reach the endpoint, then come back and swap
`VLLM_BASE_URL` above to the public hostname for the rest of the notebook).
See [`docs/deploy-vllm.md` → Access from outside the cluster (VPN)](../docs/deploy-vllm.md#access-from-outside-the-cluster-vpn)
for details.


In [ ]:
from openai import OpenAI
base_url = "https://qwen3--vl--32b--instruct-awq-runai-shared-models.deepthought.doit.wisc.edu/v1/"
client = OpenAI(
    base_url=base_url,
    api_key="not-used",
)

resp = client.chat.completions.create(
    model="QuantTrio/Qwen3-VL-32B-Instruct-AWQ",
    messages=[{"role": "user", "content": "In a single sentence, share with me the entire life story of Winnie the Pooh."}],
    max_tokens=100,
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(resp.choices[0].message.content)


## 2. Pull in the shared pipeline pieces

The notebook reuses three things from `ocr_app/scripts/` so we don't
duplicate logic that the production batch script already has:

- `chunk_page_ranges` — splits an N-page document into overlapping ranges.
- `build_chunk_messages` — builds the multi-image VLM prompt for a chunk
  (handles sliding-window context, pinned cover page, forward-context hint).
- `merge_chunks` — stitches per-chunk extractions back into one doc-level
  JSON, deduping cross-chunk tables, narratives, stakeholders, etc.
- `DOC_SYNTHESIS_PROMPT` — the per-chunk extraction prompt, shared with
  `ocr_app/scripts/ocr_server.py` so the notebook and the deployed server
  ask the model the exact same question.

Two small wrappers stay in the notebook (`run_vlm`, `parse_vlm_json`)
because they're how the demo talks directly to the vLLM endpoint — useful
for teaching, but not in `scripts/` since production goes through the
HTTP layer.


In [ ]:
import sys
from pathlib import Path

# Locate repo root regardless of where the notebook is opened from
# (cloned working copy vs. /ocr/repo symlink to a tarball).
_candidates = [Path.cwd().parents[1], Path.cwd().parent, Path("/ocr/repo")]
REPO_ROOT = next((p for p in _candidates if (p / "ocr_app" / "scripts" / "merge.py").exists()), None)
assert REPO_ROOT is not None, f"Could not find ocr_app/scripts/. Tried: {_candidates}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

from ocr_app.scripts.chunk_extract import chunk_page_ranges, build_chunk_messages
from ocr_app.scripts.merge import merge_chunks
from ocr_app.scripts.doc_prompt import DOC_SYNTHESIS_PROMPT
print("Imports OK.")


### Notebook-side wrappers: `run_vlm` and `parse_vlm_json`

`run_vlm` is the thin streaming OpenAI-compatible client this notebook
uses to talk to vLLM. We stream because the gateway has a ~3-minute idle
timeout that kills non-streaming requests for dense pages.

`parse_vlm_json` strips markdown fences and tolerates the occasional
malformed tail. With vLLM's guided JSON decoding, parse failures are rare
in practice.


In [ ]:
import base64, io, json
from PIL import Image


def _encode_image_b64(image: Image.Image) -> str:
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"


def run_vlm(messages: list, max_tokens: int = VLLM_MAX_TOKENS) -> str:
    """Stream a chat completion from the vLLM endpoint."""
    # Convert our internal {type:"image", image:...} to OpenAI image_url shape.
    oai_messages = []
    for msg in messages:
        content = []
        for part in msg["content"]:
            if part["type"] == "text":
                content.append({"type": "text", "text": part["text"]})
            elif part["type"] == "image":
                content.append({"type": "image_url", "image_url": {"url": part["image"]}})
        oai_messages.append({"role": msg["role"], "content": content})

    payload = {
        "model": MODEL_NAME,
        "messages": oai_messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": True,
        "stream_options": {"include_usage": True},
        # Force valid JSON at the token level — eliminates whole classes
        # of cleanup work (curly quotes, JS-style comments, trailing prose).
        "response_format": {"type": "json_object"},
    }
    parts = []
    with httpx.stream(
        "POST",
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=httpx.Timeout(connect=30.0, read=720.0, write=60.0, pool=30.0),
    ) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            event = json.loads(data)
            for choice in event.get("choices") or []:
                piece = (choice.get("delta") or {}).get("content")
                if piece:
                    parts.append(piece)
    return "".join(parts)


def parse_vlm_json(raw: str):
    """Parse a VLM response. Returns (dict_or_None, error_or_None)."""
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        # Last-ditch: trim to the outermost balanced object.
        first, last = cleaned.find("{"), cleaned.rfind("}")
        if first != -1 and last > first:
            try:
                return json.loads(cleaned[first:last + 1]), None
            except json.JSONDecodeError:
                pass
        return None, str(e)


print("Wrappers ready.")


## 3. The extraction prompt

`DOC_SYNTHESIS_PROMPT` is the heart of the system: it tells the VLM what
schema to emit per chunk (narratives verbatim, three table classifications,
stakeholders, addresses, signatures, watermarks, etc.) and how to flag
content that continues across chunk boundaries.

Tweaking the prompt is the highest-leverage change you can make to the
output. Edit it in `ocr_app/scripts/doc_prompt.py`, restart the kernel,
and the change flows through both this notebook and the production
batch script.


In [ ]:
print(f"Prompt length: {len(DOC_SYNTHESIS_PROMPT)} chars")
print()
print(DOC_SYNTHESIS_PROMPT[:1500])
print("...")


## 4. Pick a sample document

Upload PDFs to `/ocr/` from the JupyterLab file browser. We grab the
first one for the live walkthrough; the batch step at the bottom processes
everything in the folder.


In [ ]:
ocr_dir = Path("/ocr")
pdfs = sorted(ocr_dir.glob("*.pdf"))
assert pdfs, "No PDFs in /ocr/. Upload one first."

for p in pdfs:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

DOC_PATH = pdfs[0]
print(f"\nUsing: {DOC_PATH.name}")


## 5. Render the document as page images

Every page goes to the VLM as an image — that's how we capture layout,
tables, signatures, and watermarks that pure-text extraction would miss.
We render at 2x scale (~144 DPI), which is the sweet spot for dense
grant-admin tables: high enough that small numbers stay legible, low
enough that input tokens don't blow up.


In [ ]:
import fitz

doc = fitz.open(str(DOC_PATH))
page_images = []
for i, page in enumerate(doc):
    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
    page_images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
doc.close()

print(f"{DOC_PATH.name}: {len(page_images)} page(s)")
print(f"First page: {page_images[0].width}x{page_images[0].height}")
display(page_images[0].resize((page_images[0].width // 3, page_images[0].height // 3)))


## 6. Extract a single chunk (fast feedback loop)

Before kicking off a full batch, run the first few pages through the
pipeline so we can eyeball the output. We use the same `build_chunk_messages`
the batch loop uses — this is literally the production code path with a
chunk size of 1.


In [ ]:
import time

CHUNK_PAGES = page_images[:min(3, len(page_images))]

messages = build_chunk_messages(
    images=CHUNK_PAGES,
    prompt=DOC_SYNTHESIS_PROMPT,
    encode_image_b64=_encode_image_b64,
    filename=DOC_PATH.name,
    is_first_chunk=True,
    is_last_chunk=(len(CHUNK_PAGES) == len(page_images)),
    first_pdf_page=1,
)

t0 = time.time()
raw = run_vlm(messages)
print(f"VLM response: {len(raw)} chars in {time.time() - t0:.1f}s")

parsed, err = parse_vlm_json(raw)
assert parsed is not None, f"Parse failed: {err}"

print(f"\nTop-level fields: {list(parsed.keys())}")
print(f"  one_sentence_summary: {parsed.get('one_sentence_summary', '')[:120]}")
print(f"  tables: {len(parsed.get('tables') or [])}")
print(f"  narrative_responses: {len(parsed.get('narrative_responses') or [])}")
print(f"  stakeholders: {len(parsed.get('stakeholders') or [])}")
print(f"  addresses: {len(parsed.get('addresses') or [])}")
print(f"  document_tags: {parsed.get('document_tags') or []}")


## 7. Filename metadata parser

Grant-admin docs follow a structured filename convention:

```
A_RSP Public_AWD-001064_3718296725__AWD00000002__A_RSP_Award_Notice_of_Award.pdf
└Drawer ┘ └AwardID──┘ └F2────┘  └F4───────┘  └DocumentType─────────────────┘
                                ↑ double underscore = empty field
```

We pull these into a `FileNameMetaData` block on the doc-level output.
Useful for downstream cataloging — and a good reminder that some metadata
is cheaper to extract from the filename than from the page.


In [ ]:
import re


def parse_filename(filename: str) -> dict:
    """Parse the structured grant-admin filename. Double `__` = empty field."""
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip(".")

    awd = re.search(r"_AWD-", stem)
    if not awd:
        return {"Drawer": stem, "AwardID": "", "Field2": "", "Field3": "",
                "Field4": "", "Field5": "", "DocumentType": stem,
                "DocumentTypeShort": stem, "FileType": ext}

    drawer = stem[:awd.start()]
    parts = stem[awd.start() + 1:].split("_")
    fields = parts[1:5] + [""] * max(0, 5 - len(parts[1:5]))
    doc_type = "_".join(parts[5:]) if len(parts) > 5 else ""
    short = doc_type
    for cat in ("_Award_", "_Budget_", "_Report_", "_Agreement_", "_Proposal_"):
        i = doc_type.find(cat)
        if i >= 0:
            short = doc_type[i + len(cat):]
            break

    return {"Drawer": drawer, "AwardID": parts[0],
            "Field2": fields[0], "Field3": fields[1],
            "Field4": fields[2], "Field5": fields[3],
            "DocumentType": doc_type, "DocumentTypeShort": short,
            "FileType": ext}


for k, v in parse_filename(DOC_PATH.name).items():
    print(f"  {k}: {v!r}")


## 8. Batch all PDFs in `/ocr/`

The full batch loop:

1. Render each PDF.
2. Split into overlapping chunks (`chunk_page_ranges`).
3. Extract each chunk, parse its JSON, cache to disk so a crash is
   recoverable. Forward chunk 1's `document_details` to later chunks
   so the model has consistent doc-level grounding.
4. **Merge** chunks into one doc-level JSON via `merge_chunks` — this is
   where cross-chunk tables get stitched back together and duplicate
   stakeholders get deduped.
5. **Pass 2**: a small text-only VLM call that takes the per-chunk
   summaries and produces a single `one_sentence_summary` for the doc
   plus a `potential_issues` list (lint hits the merge couldn't see).


In [ ]:
SYNTHESIS_PROMPT = """You are given a MERGED doc-level JSON from a
multi-page grant-administration document. Pass 1 already extracted
tables, narratives, stakeholders, and addresses chunk-by-chunk and a
deterministic merge has stitched cross-chunk spans and deduped items.
Your ONLY job is to fill two summary fields and optionally flag issues.

The merged JSON includes a ``chunks`` array. Each entry has a brief
per-chunk ``extracted.one_sentence_summary`` and
``extracted.confidence_narrative`` — USE those as the raw material for
your doc-level summaries. Do not re-read the full tables/narratives.

Return a JSON object with EXACTLY these fields:
{
  "one_sentence_summary": "<one-sentence doc-level summary>",
  "confidence_narrative": "<3–5 sentences summarizing extraction quality>",
  "potential_issues": ["<optional flags for merge oddities>"]
}

Output ONLY valid JSON. No markdown fences."""


In [ ]:
def run_pass2(merged: dict) -> dict | None:
    """Compact the merged JSON, ask the VLM for a doc-level summary."""
    chunks = [
        {
            "page_range": [c.get("page_start"), c.get("page_end")],
            "one_sentence_summary": (c.get("extracted") or {}).get("one_sentence_summary", ""),
            "confidence_narrative": (c.get("extracted") or {}).get("confidence_narrative", ""),
        }
        for c in (merged.get("chunks") or [])
    ]
    compact = {
        "document_details": merged.get("document_details") or {},
        "document_tags": merged.get("document_tags") or [],
        "item_counts": {
            "tables": len(merged.get("tables") or []),
            "narrative_responses": len(merged.get("narrative_responses") or []),
            "stakeholders": len(merged.get("stakeholders") or []),
            "addresses": len(merged.get("addresses") or []),
        },
        "potential_issues": list(merged.get("potential_issues") or []),
        "chunks": chunks,
    }
    body = json.dumps(compact, indent=1, ensure_ascii=False)
    msgs = [{"role": "user", "content": [
        {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{body}"}
    ]}]
    raw = run_vlm(msgs, max_tokens=6000)
    parsed, _ = parse_vlm_json(raw)
    return parsed


# ── Tunables ────────────────────────────────────────────────────────
PDF_DIR = Path("/ocr")
OUT_DIR = Path("/ocr/vlm_output")
OUT_DIR.mkdir(exist_ok=True)
MAX_PAGES_PER_CHUNK = 20
CHUNK_OVERLAP = 1
SKIP_EXISTING = True

pdf_files = sorted(PDF_DIR.glob("*.pdf"), key=lambda p: p.stat().st_size, reverse=True)
print(f"{len(pdf_files)} PDF(s) to process")

for pdf_path in pdf_files:
    out_path = OUT_DIR / f"{pdf_path.stem}_extracted.json"
    chunks_dir = OUT_DIR / f"{pdf_path.stem}_chunks"

    if SKIP_EXISTING and out_path.exists():
        print(f"SKIP (already done): {pdf_path.name}")
        continue

    print(f"\n=== {pdf_path.name} ===")
    chunks_dir.mkdir(exist_ok=True)

    doc = fitz.open(str(pdf_path))
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()

    ranges = chunk_page_ranges(len(images), MAX_PAGES_PER_CHUNK, CHUNK_OVERLAP)
    print(f"  {len(images)} pages → {len(ranges)} chunk(s)")

    chunk_records = []
    forward_ctx = None
    for ci, (start, end) in enumerate(ranges):
        cache_file = chunks_dir / f"chunk_{ci+1:03d}.json"
        if cache_file.exists():
            chunk_records.append(json.loads(cache_file.read_text()))
            print(f"  chunk {ci+1}: cached")
            continue

        msgs = build_chunk_messages(
            images=images[start:end],
            prompt=DOC_SYNTHESIS_PROMPT,
            encode_image_b64=_encode_image_b64,
            filename=pdf_path.name,
            is_first_chunk=(ci == 0),
            is_last_chunk=(ci == len(ranges) - 1),
            pinned_images=[(images[0], "Page 1 (cover)")] if ci > 0 else None,
            forward_context=forward_ctx if ci > 0 else None,
            first_pdf_page=start + 1,
        )
        t0 = time.time()
        raw = run_vlm(msgs)
        elapsed = time.time() - t0
        extracted, err = parse_vlm_json(raw)
        if extracted is None:
            print(f"  chunk {ci+1}: PARSE FAILED ({err}) — skipping")
            continue

        record = {
            "chunk_index": ci, "page_start": start, "page_end": end,
            "experiment": {
                "model": MODEL_NAME, "elapsed_ms": round(elapsed * 1000, 1),
                "max_pages_per_chunk": MAX_PAGES_PER_CHUNK,
                "chunk_overlap": CHUNK_OVERLAP, "render_scale": 2.0,
            },
            "extracted": extracted,
        }
        cache_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
        chunk_records.append(record)
        if ci == 0:
            forward_ctx = {
                "document_details": extracted.get("document_details") or {},
                "document_tags": extracted.get("document_tags") or [],
                "one_sentence_summary": extracted.get("one_sentence_summary") or "",
            }
        print(f"  chunk {ci+1}/{len(ranges)} (pages {start+1}-{end}): {elapsed:.1f}s")

    if not chunk_records:
        print(f"  no chunks succeeded — skipping merge")
        continue

    merged = merge_chunks(chunk_records, extraction_prompt=DOC_SYNTHESIS_PROMPT)
    fmeta = parse_filename(pdf_path.name)
    fmeta["page_count"] = len(images)
    merged = {"filename": pdf_path.name, "file_metadata": fmeta, **merged}
    print(f"  merged: {len(merged.get('tables') or [])} tables, "
          f"{len(merged.get('narrative_responses') or [])} narratives, "
          f"{len(merged.get('stakeholders') or [])} stakeholders")

    synthesis = run_pass2(merged)
    if synthesis:
        if synthesis.get("one_sentence_summary"):
            merged["one_sentence_summary"] = synthesis["one_sentence_summary"]
        if synthesis.get("confidence_narrative"):
            merged["confidence_narrative"] = synthesis["confidence_narrative"]
        existing = list(merged.get("potential_issues") or [])
        for note in synthesis.get("potential_issues") or []:
            if note and note not in existing:
                existing.append(note)
        merged["potential_issues"] = existing
        print(f"  pass 2: {merged['one_sentence_summary'][:90]!r}")

    final = {k: v for k, v in merged.items() if k != "chunks"}
    out_path.write_text(json.dumps(final, indent=2, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}")

print("\nBatch complete.")


## 9. Inspect a result

Look at the doc-level JSON produced by the batch. Useful fields to sanity
check:

- `one_sentence_summary` — the pass-2 doc summary
- `file_metadata` — parsed filename
- `tables` / `narrative_responses` / `stakeholders` / `addresses` — the
  extracted content
- `potential_issues` — merge lint + pass-2 flags


In [ ]:
out_files = sorted(OUT_DIR.glob("*_extracted.json"))
assert out_files, f"No outputs in {OUT_DIR}/. Run the batch cell first."
sample = json.loads(out_files[0].read_text())

print(f"=== {sample['filename']} ===")
print(f"summary: {sample.get('one_sentence_summary', '')}")
print(f"\ndocument_details: {json.dumps(sample.get('document_details', {}), indent=2)}")
print(f"\ndocument_tags: {sample.get('document_tags', [])}")
print(f"\ncounts:")
print(f"  tables:              {len(sample.get('tables') or [])}")
print(f"  narrative_responses: {len(sample.get('narrative_responses') or [])}")
print(f"  stakeholders:        {len(sample.get('stakeholders') or [])}")
print(f"  addresses:           {len(sample.get('addresses') or [])}")
issues = sample.get("potential_issues") or []
print(f"\npotential_issues: {len(issues)}")
for issue in issues[:5]:
    print(f"  • {issue}")
